# 01 Baseline: Booking.com reviews (EDA, sentences, aspects, baselines, labelling sheet)

Dataset: Booking.com "515K Hotel Reviews Data in Europe" (Kaggle). Save the CSV as `data/raw/Hotel_Reviews.csv`.

**Colab:** the first code cell mounts Drive and moves into the `notebooks` folder (it does nothing when you run locally). After changing `src/pipeline.py`, use Runtime > Restart session so Python reloads it.

In [ ]:
# Colab only: mount Drive and move into the notebooks folder. Skipped when running locally.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/hotel-review-aspect-sentiment/notebooks')
except ImportError:
    pass

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

from src.pipeline import (load_reviews, score_to_sentiment, split_sentences,
                          add_difficulty_flags, sample_for_labelling)

ROOT = Path.cwd().parent
DATA_PATH = ROOT / "data" / "raw" / "Hotel_Reviews.csv"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
(ROOT / "data" / "labelled").mkdir(parents=True, exist_ok=True)
SEED = 42
N_REVIEWS = 20000   # random sample of the 515K reviews, so every run stays fast

## 1. Load a random sample and inspect
The loader keeps only the positive text, negative text and reviewer score (plus hotel name). The placeholders `No Positive` / `No Negative` become empty.

In [ ]:
df = load_reviews(DATA_PATH, n_sample=N_REVIEWS, seed=SEED)
print(df.shape)
df[["hotel", "score", "positive", "negative"]].head()

In [ ]:
df["n_words"] = df["review"].str.split().map(len)
print(df["n_words"].describe())
print("Reviews with no positive text:", round((df["positive"] == "").mean(), 3),
      "| with no negative text:", round((df["negative"] == "").mean(), 3))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
df["score"].round().value_counts().sort_index().plot.bar(ax=ax[0], title="Reviewer score (rounded)")
df["n_words"].clip(upper=300).plot.hist(bins=40, ax=ax[1], title="Review length (words, clipped at 300)")
plt.tight_layout()
plt.savefig(OUT / "eda_scores_lengths.png", dpi=150)
plt.show()

## 2. Review-level labels and baseline
Assumption: score >= 8 positive, 6 to <8 neutral, < 6 negative. This is the 'standard sentiment baseline': one label per review, so it cannot separate aspects.

In [ ]:
df["sentiment"] = df["score"].map(score_to_sentiment)
print(df["sentiment"].value_counts(normalize=True).round(3))

X_train, X_test, y_train, y_test = train_test_split(
    df["review"], df["sentiment"], test_size=0.2, random_state=SEED, stratify=df["sentiment"])
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(vec.fit_transform(X_train), y_train)
pred = clf.predict(vec.transform(X_test))
print(classification_report(y_test, pred, digits=3))
review_macro_f1 = f1_score(y_test, pred, average="macro")
review_macro_f1

## 3. Comments, aspect tagging and difficulty flags
**This dataset has no punctuation**, so the splitter cannot find sentence boundaries and each text box is one short comment (the code still calls it a "sentence"). Each comment keeps the box it was written in (`field` = positive / negative): a free but noisy label.

In [ ]:
sents = split_sentences(df)
flagged = add_difficulty_flags(sents)
print(len(sents), "sentences from", len(df), "reviews")
per_box = flagged.groupby(["review_id", "field"]).size()
print("Sentences per text box: mean", round(per_box.mean(), 2), "| max", per_box.max(), "(about 1 means no sentence boundaries were found)")
print("Sentences with at least one aspect:", round((flagged["n_aspects"] >= 1).mean(), 3))
print("Negation:", round(flagged["has_negation"].mean(), 3),
      "| Contrast:", round(flagged["has_contrast"].mean(), 3),
      "| Multi-aspect:", round(flagged["is_multi_aspect"].mean(), 3))
flagged.sample(8, random_state=SEED)[["sentence", "field", "aspects"]]

In [ ]:
aspect_counts = flagged.explode("aspects")["aspects"].value_counts()
aspect_counts.plot.barh(title="Sentences per aspect (keyword lexicon)")
plt.tight_layout(); plt.savefig(OUT / "aspect_counts.png", dpi=150); plt.show()
aspect_counts

## 4. Sentence-level baseline using weak labels (positive box vs negative box)
Train/test are split **by review** so sentences from one review never appear in both.
These labels are noisy (a sentence in the negative box is not always negative), so the per-category numbers are only a rough early signal. The real evaluation uses your hand labels.

In [ ]:
sent = flagged[flagged["n_words"] >= 3].copy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, te_idx = next(gss.split(sent, groups=sent["review_id"]))
tr, te = sent.iloc[tr_idx], sent.iloc[te_idx]

vec2 = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
clf2 = LogisticRegression(max_iter=1000, class_weight="balanced")
clf2.fit(vec2.fit_transform(tr["sentence"]), tr["field"])
te = te.assign(pred=clf2.predict(vec2.transform(te["sentence"])))
print(classification_report(te["field"], te["pred"], digits=3))

rows = [("all", f1_score(te["field"], te["pred"], average="macro"), len(te))]
for name, mask in [("negation", te["has_negation"]), ("contrast", te["has_contrast"]), ("multi_aspect", te["is_multi_aspect"])]:
    sub = te[mask]
    if len(sub) > 0:
        rows.append((name, f1_score(sub["field"], sub["pred"], average="macro"), len(sub)))
res = pd.DataFrame(rows, columns=["subset", "macro_f1", "n_sentences"])
res.to_csv(OUT / "sentence_baseline_by_category.csv", index=False)
res

## 5. Export the hand-labelling sheet (about 200 sentence-aspect rows)
One row per (sentence, aspect) pair. A "sentence" here is usually a whole short comment, because the text has no punctuation. For each row fill in:
- `gold_sentiment`: the sentiment expressed toward **that aspect** in that sentence: `positive`, `negative`, `neutral`, or `not_applicable` (the keyword is not really about that aspect, or the text is not English / unclear).
- `difficulty`: `negation`, `contrast`, `multi_aspect` or `plain`.

The box each sentence came from is hidden on purpose. Sampling is deterministic, so with the same `N_REVIEWS` and `SEED` you can join it back later with `review_id` + `sent_id`.

In [ ]:
sheet = sample_for_labelling(flagged, n_pairs=200, seed=SEED)
sheet.to_csv(ROOT / "data" / "labelled" / "labelling_sheet.csv", index=False)
print(len(sheet), "aspect rows from", sheet[["review_id", "sent_id"]].drop_duplicates().shape[0], "sentences")
print(sheet["sample_group"].value_counts())
sheet.head(10)

## Next steps
1. Label `data/labelled/labelling_sheet.csv` (open it in Sheets or Excel), save as `labelled_pairs.csv`.
2. Add the transformer model (Hugging Face sentiment pipeline) and compare per difficulty category on your hand labels.
3. Build the demo (notebook cell or Gradio app).